In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Helper to compute basic stats (raw unix seconds min/max and row count). Also add parsed datetime (UTC).
def file_stats(df, name):
    if 'timestamp' in df.columns and len(df) > 0:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
    else:
        min_ts = None
        max_ts = None
    # add parsed datetime column for internal use (UTC)
    if 'timestamp' in df.columns:
        df['_ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    return {'file': name, 'rows': len(df), 'min_timestamp': min_ts, 'max_timestamp': max_ts}

# Compute file-level stats
metric_stats = file_stats(metric, 'metric.csv')
trace_stats = file_stats(trace, 'trace.csv')
log_stats = file_stats(log, 'log.csv')
error_logs_stats = file_stats(error_logs, 'error_logs.csv')

file_stats_df = pd.DataFrame([metric_stats, trace_stats, log_stats, error_logs_stats])

# --- Metric aggregation across full series (no filtering) ---
# Ensure value column is numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Group-level count and mean
grp_cnt_mean = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].agg(count='count', mean='mean').reset_index()

# Group-level global p95 (quantile computed over full available series)
grp_p95 = metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95).reset_index().rename(columns={'value':'p95'})

# Merge counts/means with p95
metric_agg = grp_cnt_mean.merge(grp_p95, on=['cmdb_id','kpi_name'], how='left')

# Formatting: round mean and p95 for compact display
metric_agg['mean'] = metric_agg['mean'].round(6)
metric_agg['p95'] = metric_agg['p95'].round(6)

# Top 50 by count descending
metric_agg_top50 = metric_agg.sort_values('count', ascending=False).head(50)[['cmdb_id','kpi_name','count','mean','p95']]

# --- Trace: unique trace_name and counts (top 50) ---
if 'trace_name' in trace.columns:
    trace_counts = trace.groupby('trace_name', dropna=False).size().reset_index(name='count').sort_values('count', ascending=False).head(50)
else:
    trace_counts = pd.DataFrame(columns=['trace_name','count'])

trace_counts_top50 = trace_counts

# --- Log: unique log_name and counts (top 50) ---
if 'log_name' in log.columns:
    log_counts = log.groupby('log_name', dropna=False).size().reset_index(name='count').sort_values('count', ascending=False).head(50)
else:
    log_counts = pd.DataFrame(columns=['log_name','count'])

log_counts_top50 = log_counts

# Display compact summaries
file_stats_df, metric_agg_top50, trace_counts_top50, log_counts_top50

```
Out[1]:
```
Summary of the loaded telemetry (plain English):

Files and row/time ranges
- metric.csv: 1875 rows; min timestamp = 1705682000, max timestamp = 1705683000.
- trace.csv: 3784 rows; min timestamp = 1705682000, max timestamp = 1705683000.
- log.csv: 460 rows; min timestamp = 1705682000, max timestamp = 1705683000.
- error_logs.csv: 0 rows (empty).

Metric aggregation (aggregated by cmdb_id + kpi_name; statistics computed over the full series)
- Aggregation produced up to 50 component–KPI rows (top 50 by count). In the top 50, most groups have count = 25 (i.e., 25 samples per component–KPI in those top entries).
- Example top entries (rounded):
  - adservice / cpu: count=25, mean≈0.715609, p95≈0.779409
  - adservice / mem: count=25, mean≈109,572,200, p95≈109,708,300
  - adservice / diskio: count=25, mean≈63,132.65, p95≈75,839.81
  - paymentservice / mem: count=25, mean≈45,340,990, p95≈45,961,340
  - paymentservice / socket: count=25, mean=3.0, p95=3.0
- Overall: the top-50 component–KPI rows show consistent sample counts (mostly 25) with per-group mean and p95 as reported above.

Trace telemetry
- Found 44 unique trace_name values (top results shown).
- Highest-count trace entries are the "trace.from_root.*" series (each with count = 173 for row_count, error_rate, duration_p95, duration_mean).
- Other notable counts: trace.from_frontendservice.duration_mean = 125; several inter-service trace metrics range from ~49 to ~125 samples.

Log telemetry
- Two unique log_name values: log.error_count (count=230) and log.row_count (count=230). (Total log rows = 460.)

Notes
- All aggregations used the full available series (no incident-window filtering).
- error_logs.csv contains no rows.

The original code execution output of IPython Kernel is also provided below for reference:

(             file  rows  min_timestamp  max_timestamp
0      metric.csv  1875   1.705682e+09   1.705683e+09
1       trace.csv  3784   1.705682e+09   1.705683e+09
2         log.csv   460   1.705682e+09   1.705683e+09
3  error_logs.csv     0            NaN            NaN,            cmdb_id    kpi_name  count          mean           p95
0        adservice         cpu     25  7.156090e-01  7.794090e-01
1        adservice      diskio     25  6.313265e+04  7.583981e+04
2        adservice  latency-50     25  3.003000e-03  3.017000e-03
3        adservice  latency-90     25  4.616000e-03  4.636000e-03
4        adservice         mem     25  1.095722e+08  1.097083e+08
..             ...         ...    ...           ...           ...
45  paymentservice      diskio     25  0.000000e+00  0.000000e+00
46  paymentservice  latency-50     25  3.259000e-03  3.503000e-03
47  paymentservice  latency-90     25  5.711000e-03  7.479000e-03
48  paymentservice         mem     25  4.534099e+07  4.596134e+07
49  paymentservice      socket     25  3.000000e+00  3.000000e+00

[50 rows x 5 columns],                                   trace_name  count
15                 trace.from_root.row_count    173
14                trace.from_root.error_rate    173
13              trace.from_root.duration_p95    173
12             trace.from_root.duration_mean    173
4   trace.from_frontendservice.duration_mean    125
..                                       ...    ...
28    trace.to_frontendservice.duration_mean     50
35         trace.to_paymentservice.row_count     49
34        trace.to_paymentservice.error_rate     49
33      trace.to_paymentservice.duration_p95     49
32     trace.to_paymentservice.duration_mean     49

[44 rows x 2 columns],           log_name  count
0  log.error_count    230
1    log.row_count    230)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' DataFrame loaded previously.
# Ensure 'metric' has expected columns and numeric 'value'
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Components of interest
components = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# Filter metric to only these cmdb_ids for grouping calculations
metric_focused = metric[metric['cmdb_id'].isin(components)].copy()

# Parse timestamp to UTC datetime (rule 10)
if '_ts_dt' not in metric_focused.columns:
    metric_focused['_ts_dt'] = pd.to_datetime(metric_focused['timestamp'], unit='s', utc=True)

# 1) Compute global stats (over full file, per group) BEFORE any time-window filtering
grouped = metric_focused.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
global_stats = grouped.agg(count='count', mean='mean').reset_index()
# Compute p95 globally per group
p95_df = grouped.quantile(0.95).reset_index().rename(columns={'value':'p95'})
# Merge
global_stats = global_stats.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
# Round numeric columns for compactness
global_stats['mean'] = global_stats['mean'].round(6)
global_stats['p95'] = global_stats['p95'].round(6)

# 2) Filter to incident window (UTC)
window_start = pd.to_datetime('2024-01-19 16:31:57', utc=True)
window_end   = pd.to_datetime('2024-01-19 17:01:57', utc=True)
# Use parsed datetime column for filtering
window_mask = (metric_focused['_ts_dt'] >= window_start) & (metric_focused['_ts_dt'] <= window_end)
metric_window = metric_focused[window_mask].copy()

# 3) For each component-kpi compute window_count, breach_count (value >= global p95), earliest breach timestamp, window_max
# Merge global p95 into window rows to evaluate breaches
metric_window = metric_window.merge(global_stats[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')

# Mark breaches where value >= p95 (note: p95 could be NaN if group missing)
metric_window['is_breach'] = metric_window['value'] >= metric_window['p95']

# Aggregations in the window
agg_window = metric_window.groupby(['cmdb_id','kpi_name'], dropna=False).agg(
    window_count = ('value','count'),
    breach_count = ('is_breach','sum'),
    first_breach_timestamp = ('timestamp', lambda s: int(s[metric_window.loc[s.index,'is_breach']].min()) if s[metric_window.loc[s.index,'is_breach']].any() else np.nan),
    window_max = ('value','max')
).reset_index()

# There might be groups with zero rows in the window; ensure they appear with zeros/NaN
summary = global_stats.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')
summary['window_count'] = summary['window_count'].fillna(0).astype(int)
summary['breach_count'] = summary['breach_count'].fillna(0).astype(int)
summary['first_breach_timestamp'] = summary['first_breach_timestamp'].where(~summary['first_breach_timestamp'].isna(), np.nan)
summary['window_max'] = summary['window_max'].round(6)

# Select and order columns as requested
summary_df = summary[['cmdb_id','kpi_name','count','mean','p95','window_count','breach_count','first_breach_timestamp','window_max']]

# Keep compact: top 50 rows (there should be fewer)
summary_df_top50 = summary_df.sort_values(['cmdb_id','kpi_name']).head(50)

# 4) Prepare anomaly rows within window where value >= group p95, up to top 20 sorted by value desc
anomalies = metric_window[metric_window['is_breach']].copy()
# Select columns timestamp (raw unix seconds), cmdb_id, kpi_name, value
anomalies_sel = anomalies[['timestamp','cmdb_id','kpi_name','value']].sort_values('value', ascending=False).head(20)
anomalies_top20 = anomalies_sel.reset_index(drop=True)

# Final outputs (compact)
summary_df_top50, anomalies_top20

```
Out[2]:
```
Summary (analysis restricted to the five services; window = 2024-01-19 16:31:57 → 2024-01-19 17:01:57 UTC):

- Scope & counts
  - Components analyzed: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice.
  - Component–KPI groups returned: 34 rows (each group computed global stats across the full file before window filtering).
  - Most groups have global count = 25 (statistics computed over the full series as requested).

- Global statistics (per group, examples)
  - Each group includes: global count, global mean, global P95 (computed over the entire file for that group).
  - Example (rounded): checkoutservice mem — count=25, mean≈11,492,490, p95≈11,729,690; currencyservice mem — mean≈49,3xx,xxx, p95≈49,5xx,xxx (see table for all groups).

- Window-level findings (for each group we computed window_count, breach_count where value >= global P95, first_breach_timestamp (unix seconds) if any, and window_max)
  - Many groups had window_count = 25 (i.e., 25 samples present inside the incident window).
  - Most KPIs show small numbers of breaches in-window (breach_count commonly = 2 for many KPIs).
  - Memory (mem) metrics breached P95 across multiple services during the window.
  - currencyservice shows the largest mem values in-window and the top anomalies (largest absolute mem values).
  - checkoutservice and recommendationservice socket KPIs show breach_count = 25 (socket values were equal to the P95 / constant), i.e., every point in-window met the breach condition for socket.
  - currencyservice experienced very large CPU spikes in-window (values near ~50).

- Top anomaly rows within the window (value >= group P95), sorted by value desc — top 10 shown (timestamps are raw unix seconds UTC):
  1. 1705683420 — currencyservice — mem — 49,559,830
  2. 1705683300 — currencyservice — mem — 49,326,010
  3. 1705682040 — recommendationservice — mem — 45,649,920
  4. 1705682520 — recommendationservice — mem — 45,558,720
  5. 1705682160 — emailservice — mem — 43,976,500
  6. 1705682220 — emailservice — mem — 43,965,580
  7. 1705682520 — checkoutservice — mem — 11,738,660
  8. 1705683060 — checkoutservice — mem — 11,734,970
  9. 1705682580 — productcatalogservice — mem — 10,811,050
 10. 1705683420 — productcatalogservice — mem — 10,733,700

- Takeaway
  - Memory metrics across multiple services breached their group-level P95s during the window, with currencyservice.mem showing the largest in-window values. CPU spikes (currencyservice) and constant socket values (checkoutservice, recommendationservice) are also notable. Timestamps reported above are raw unix seconds (UTC).

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count          mean           p95  window_count  breach_count  first_breach_timestamp    window_max
0         checkoutservice         cpu     25  4.023760e-01  5.223210e-01            25             2              1705683180  5.477250e-01
1         checkoutservice  latency-50     25  1.488310e-01  2.141990e-01            25             2              1705682880  2.369990e-01
2         checkoutservice  latency-90     25  3.228630e-01  5.547800e-01            25             2              1705682940  6.090030e-01
3         checkoutservice         mem     25  1.149249e+07  1.172969e+07            25             2              1705682520  1.173866e+07
4         checkoutservice      socket     25  9.000000e+00  9.000000e+00            25            25              1705682040  9.000000e+00
..                    ...         ...    ...           ...           ...           ...           ...                     ...           ...
29  recommendationservice  latency-50     25  7.520000e-03  7.585000e-03            25             2              1705682880  7.646000e-03
30  recommendationservice  latency-90     25  9.650000e-03  9.741000e-03            25             2              1705682880  1.020500e-02
31  recommendationservice         mem     25  4.552201e+07  4.555428e+07            25             2              1705682040  4.564992e+07
32  recommendationservice      socket     25  4.000000e+00  4.000000e+00            25            25              1705682040  4.000000e+00
33  recommendationservice    workload     25  1.856868e+01  1.951068e+01            25             2              1705682160  1.956337e+01

[34 rows x 9 columns],      timestamp                cmdb_id  kpi_name         value
0   1705683420        currencyservice       mem  4.955983e+07
1   1705683300        currencyservice       mem  4.932601e+07
2   1705682040  recommendationservice       mem  4.564992e+07
3   1705682520  recommendationservice       mem  4.555872e+07
4   1705682160           emailservice       mem  4.397650e+07
5   1705682220           emailservice       mem  4.396558e+07
6   1705682520        checkoutservice       mem  1.173866e+07
7   1705683060        checkoutservice       mem  1.173497e+07
8   1705682580  productcatalogservice       mem  1.081105e+07
9   1705683420  productcatalogservice       mem  1.073370e+07
10  1705682820        currencyservice    diskio  4.834037e+04
11  1705682820  productcatalogservice  workload  1.483056e+02
12  1705682220  productcatalogservice  workload  1.477710e+02
13  1705682280        currencyservice  workload  8.578198e+01
14  1705682460        currencyservice  workload  8.131443e+01
15  1705683360        currencyservice       cpu  4.996998e+01
16  1705683240        currencyservice       cpu  4.983541e+01
17  1705682160  recommendationservice  workload  1.956337e+01
18  1705682580  recommendationservice  workload  1.953000e+01
19  1705682040        checkoutservice    socket  9.000000e+00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing 'trace' DataFrame loaded earlier.
# Ensure numeric value
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Components of interest
components = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# Ensure parsed datetime column (UTC)
if '_ts_dt' not in trace.columns:
    trace['_ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
else:
    # ensure it's datetime type
    trace['_ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Filter to components of interest
trace_focused = trace[trace['cmdb_id'].isin(components)].copy()

# 1) Global stats per (cmdb_id, trace_name) over full available series (no window filtering)
grp = trace_focused.groupby(['cmdb_id','trace_name'], dropna=False)['value']
global_stats = grp.agg(count='count', mean='mean').reset_index()
p95_df = grp.quantile(0.95).reset_index().rename(columns={'value':'p95'})
global_stats = global_stats.merge(p95_df, on=['cmdb_id','trace_name'], how='left')
global_stats['mean'] = global_stats['mean'].round(6)
global_stats['p95'] = global_stats['p95'].round(6)

# 2) Filter to incident window (UTC)
window_start = pd.to_datetime('2024-01-19 16:31:57', utc=True)
window_end   = pd.to_datetime('2024-01-19 17:01:57', utc=True)
window_mask = (trace_focused['_ts_dt'] >= window_start) & (trace_focused['_ts_dt'] <= window_end)
trace_window = trace_focused[window_mask].copy()

# 3) Merge global p95 into window rows to evaluate breaches
trace_window = trace_window.merge(global_stats[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='left')
trace_window['is_breach'] = trace_window['value'] >= trace_window['p95']

# 4) Window aggregations: window_count, breach_count, window_max
window_agg = trace_window.groupby(['cmdb_id','trace_name'], dropna=False).agg(
    window_count = ('value','count'),
    breach_count = ('is_breach','sum'),
    window_max = ('value','max')
).reset_index()

# 5) first_breach_timestamp per group (min raw unix timestamp where is_breach is True)
first_breach = trace_window[trace_window['is_breach']].groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'first_breach_timestamp'})

# Merge first_breach into window_agg
window_agg = window_agg.merge(first_breach, on=['cmdb_id','trace_name'], how='left')

# 6) Combine global stats with window aggregates
summary = global_stats.merge(window_agg, on=['cmdb_id','trace_name'], how='left')

# Fill missing window aggregates with zeros/NaN as appropriate
summary['window_count'] = summary['window_count'].fillna(0).astype(int)
summary['breach_count'] = summary['breach_count'].fillna(0).astype(int)
summary['first_breach_timestamp'] = summary['first_breach_timestamp'].where(~summary['first_breach_timestamp'].isna(), np.nan)
summary['window_max'] = summary['window_max'].round(6)

# Select and order columns, sort by breach_count desc then window_max desc, limit top 50
summary_df = summary[['cmdb_id','trace_name','count','mean','p95','window_count','breach_count','first_breach_timestamp','window_max']]
summary_top50 = summary_df.sort_values(['breach_count','window_max'], ascending=[False, False]).head(50)

# 7) Prepare anomaly rows within window where value >= p95, up to top 20 sorted by value desc
anomalies = trace_window[trace_window['is_breach']].copy()
anomalies_sel = anomalies[['timestamp','cmdb_id','trace_name','value']].sort_values('value', ascending=False).head(20).reset_index(drop=True)

# Final outputs (compact)
summary_top50, anomalies_sel

```
Out[3]:
```
summary = (
    "Trace analysis for checkoutservice, currencyservice, emailservice, "
    "productcatalogservice, recommendationservice (window 2024-01-19 16:31:57 → 2024-01-19 17:01:57 UTC):\n"
    "- Method: computed global count/mean/P95 per (cmdb_id, trace_name) over full file, then evaluated window_count, "
    "breach_count (value >= group P95), first_breach_timestamp (unix seconds), and window_max inside the window.\n"
    "- Key findings:\n"
    "  * Many 'error_rate' trace series have global P95 = 0 and therefore show breach_count = full window (all points == 0 ≥ 0).\n"
    "  * The largest, high-severity breaches are for row_count traces (large traffic spikes). Top anomalies (value desc) include:\n"
    "    - productcatalogservice, trace.from_frontendservice.row_count = 3850 at 1705682760\n"
    "    - productcatalogservice, trace.from_frontendservice.row_count = 3837 at 1705682580\n"
    "    - currencyservice, trace.from_frontendservice.row_count = 2378 at 1705682280\n"
    "    - currencyservice, trace.from_frontendservice.row_count = 2357 at 1705682340\n"
    "  * Several services show high window_max for 'row_count' and other load-related traces (recommendationservice and checkoutservice also show large row_count values at specific timestamps).\n"
    "- Conclusion: the dominant anomalies in the incident window are large row_count (traffic/load) spikes on productcatalogservice, currencyservice, recommendationservice and checkoutservice. Many error_rate series report P95=0 so appear as full-window 'breaches' but those reflect the P95 computation rather than an error-rate increase."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                trace_name  count       mean        p95  window_count  breach_count  first_breach_timestamp  window_max
2   checkoutservice     trace.from_checkoutservice.error_rate     25   0.000000   0.000000            25            25              1705682040    0.000000
6   checkoutservice     trace.from_frontendservice.error_rate     25   0.000000   0.000000            25            25              1705682040    0.000000
10  checkoutservice                trace.from_root.error_rate     25   0.000000   0.000000            25            25              1705682040    0.000000
14  checkoutservice       trace.to_checkoutservice.error_rate     25   0.000000   0.000000            25            25              1705682040    0.000000
18  checkoutservice       trace.to_currencyservice.error_rate     25   0.000000   0.000000            25            25              1705682040    0.000000
..              ...                                       ...    ...        ...        ...           ...           ...                     ...         ...
47     emailservice      trace.from_checkoutservice.row_count     25  26.880000  36.800000            25             2              1705682700   37.000000
5   checkoutservice   trace.from_frontendservice.duration_p95     25   0.327468   0.845506            25             2              1705682940    0.907642
4   checkoutservice  trace.from_frontendservice.duration_mean     25   0.160002   0.263766            25             2              1705682940    0.313654
9   checkoutservice              trace.from_root.duration_p95     25   0.080176   0.094260            25             2              1705682880    0.094625
8   checkoutservice             trace.from_root.duration_mean     25   0.027926   0.043042            25             2              1705682940    0.043498

[50 rows x 9 columns],      timestamp                cmdb_id                                  trace_name   value
0   1705682760  productcatalogservice        trace.from_frontendservice.row_count  3850.0
1   1705682580  productcatalogservice        trace.from_frontendservice.row_count  3837.0
2   1705682280        currencyservice        trace.from_frontendservice.row_count  2378.0
3   1705682340        currencyservice        trace.from_frontendservice.row_count  2357.0
4   1705682760  productcatalogservice  trace.from_recommendationservice.row_count   597.0
5   1705682760  recommendationservice        trace.from_frontendservice.row_count   597.0
6   1705682760  recommendationservice  trace.from_recommendationservice.row_count   597.0
7   1705682760  recommendationservice    trace.to_productcatalogservice.row_count   597.0
8   1705682760  recommendationservice    trace.to_recommendationservice.row_count   597.0
9   1705682580  recommendationservice  trace.from_recommendationservice.row_count   595.0
10  1705682580  productcatalogservice  trace.from_recommendationservice.row_count   595.0
11  1705682580  recommendationservice    trace.to_productcatalogservice.row_count   595.0
12  1705682580  recommendationservice        trace.from_frontendservice.row_count   595.0
13  1705682580  recommendationservice    trace.to_recommendationservice.row_count   595.0
14  1705683120        checkoutservice        trace.from_checkoutservice.row_count   321.0
15  1705683120        checkoutservice          trace.to_checkoutservice.row_count   321.0
16  1705682700        checkoutservice        trace.from_checkoutservice.row_count   310.0
17  1705682700        checkoutservice          trace.to_checkoutservice.row_count   310.0
18  1705683120        checkoutservice                   trace.from_root.row_count   160.0
19  1705682700        checkoutservice                   trace.from_root.row_count   149.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse / ensure variables
components = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# Ensure log DataFrame exists and numeric 'value'
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Ensure parsed datetime UTC for log
log['_ts_dt'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
window_start = pd.to_datetime('2024-01-19 16:31:57', utc=True)
window_end   = pd.to_datetime('2024-01-19 17:01:57', utc=True)

# Filter to components of interest and window
log_window = log[log['cmdb_id'].isin(components) & (log['_ts_dt'] >= window_start) & (log['_ts_dt'] <= window_end)].copy()

# If no rows, produce empty summaries gracefully
if log_window.shape[0] == 0:
    log_summary_top50 = pd.DataFrame(columns=['cmdb_id','log_name','window_count','sum','mean','max','ts_of_max'])
    log_top20 = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
else:
    # Group aggregations: window_count, sum(value), mean(value), max(value)
    grp = log_window.groupby(['cmdb_id','log_name'], dropna=False)
    agg_df = grp['value'].agg(window_count='count', sum='sum', mean='mean', max='max').reset_index()
    # Earliest timestamp of max: compute per-group
    def earliest_ts_of_max(sub):
        if sub.empty:
            return np.nan
        max_val = sub['value'].max()
        ts = sub.loc[sub['value'] == max_val, 'timestamp'].min()
        if pd.isna(ts):
            return np.nan
        return int(ts)
    ts_max = grp.apply(earliest_ts_of_max).reset_index().rename(columns={0:'ts_of_max'})
    log_summary = agg_df.merge(ts_max, on=['cmdb_id','log_name'], how='left')
    # Format numeric columns compactly
    log_summary['mean'] = log_summary['mean'].round(6)
    log_summary['sum'] = log_summary['sum'].round(6)
    log_summary['max'] = log_summary['max'].round(6)
    # Select and order columns
    log_summary = log_summary[['cmdb_id','log_name','window_count','sum','mean','max','ts_of_max']]
    # Limit to top 50 rows (sort by window_count desc then sum desc for compactness)
    log_summary_top50 = log_summary.sort_values(['window_count','sum'], ascending=[False, False]).head(50).reset_index(drop=True)

    # Top 20 log rows in window sorted by value desc
    log_top20 = log_window[['timestamp','cmdb_id','log_name','value']].sort_values('value', ascending=False).head(20).reset_index(drop=True)

# Final compact outputs
log_summary_top50, log_top20

```
Out[4]:
```
Summary (logs for checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice during 2024-01-19 16:31:57 → 2024-01-19 17:01:57 UTC):

- Scope: analyzed log.csv rows for the five services inside the incident window.

- Group-level results (compact):
  - currencyservice / log.row_count: window_count=25, sum=55,721, mean≈2,228.84, max=2,477 (ts_of_max = 1705682280 UTC).
  - recommendationservice / log.row_count: window_count=25, sum=13,358, mean≈534.32, max=597 (ts_of_max = 1705682760 UTC).
  - checkoutservice / log.row_count: window_count=25, sum=2,016, mean≈80.64, max=111 (ts_of_max = 1705682700 UTC).
  - emailservice / log.row_count: window_count=25, sum=672, mean≈26.88, max=37 (ts_of_max = 1705682700 UTC).
  - For all five services, log.error_count rows exist (window_count=25) but sum=0 and max=0 (ts_of_max = 1705682040 for the shown groups) — i.e., no error_count spikes in the window.

- Top individual log rows in the window (by value, raw unix timestamps shown):
  - currencyservice / log.row_count entries dominate the top 20 (highest values include 2477 at 1705682280, 2458 at 1705682340, 2424 at 1705682580, … down to ~2238).
  - recommendationservice and others appear further down (max 597 for recommendationservice).

- Takeaway:
  - The primary log-level signal is very large log.row_count (traffic/load) on currencyservice during the window — far larger than the other services. recommendationservice shows a secondary spike. There were no log.error_count increases (all zeros). Timestamps reported are raw unix seconds (UTC).

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  window_count    sum     mean   max   ts_of_max
0        currencyservice    log.row_count            25  55721  2228.84  2477  1705682280
1  recommendationservice    log.row_count            25  13358   534.32   597  1705682760
2        checkoutservice    log.row_count            25   2016    80.64   111  1705682700
3           emailservice    log.row_count            25    672    26.88    37  1705682700
4        checkoutservice  log.error_count            25      0     0.00     0  1705682040
5        currencyservice  log.error_count            25      0     0.00     0  1705682040
6           emailservice  log.error_count            25      0     0.00     0  1705682040
7  recommendationservice  log.error_count            25      0     0.00     0  1705682040,      timestamp          cmdb_id       log_name  value
0   1705682280  currencyservice  log.row_count   2477
1   1705682340  currencyservice  log.row_count   2458
2   1705682580  currencyservice  log.row_count   2424
3   1705682220  currencyservice  log.row_count   2420
4   1705682460  currencyservice  log.row_count   2407
5   1705682100  currencyservice  log.row_count   2394
6   1705682400  currencyservice  log.row_count   2382
7   1705683360  currencyservice  log.row_count   2361
8   1705682520  currencyservice  log.row_count   2338
9   1705683180  currencyservice  log.row_count   2328
10  1705682880  currencyservice  log.row_count   2326
11  1705682820  currencyservice  log.row_count   2316
12  1705683420  currencyservice  log.row_count   2314
13  1705682700  currencyservice  log.row_count   2311
14  1705683000  currencyservice  log.row_count   2295
15  1705683120  currencyservice  log.row_count   2294
16  1705682160  currencyservice  log.row_count   2269
17  1705683300  currencyservice  log.row_count   2266
18  1705682760  currencyservice  log.row_count   2262
19  1705682640  currencyservice  log.row_count   2238)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing 'trace' DataFrame and components list
components = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# Ensure numeric and datetime
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['_ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Filter to components of interest and trace_name containing any of the target substrings
targets = ['row_count','duration_mean','duration_p95','error_rate']
pattern = '|'.join(map(repr, targets)).replace("'", "")  # simple join (no special regex chars in targets)
mask_comp = trace['cmdb_id'].isin(components)
mask_name = trace['trace_name'].str.contains('row_count|duration_mean|duration_p95|error_rate', na=False)
trace_filtered = trace[mask_comp & mask_name].copy()

# 1) Compute global P95 per (cmdb_id, trace_name) using the full available series (no window filtering)
grp_full = trace_filtered.groupby(['cmdb_id','trace_name'], dropna=False)['value']
p95_df = grp_full.quantile(0.95).reset_index().rename(columns={'value':'p95'})

# 2) Filter trace rows to the incident window (UTC)
window_start = pd.to_datetime('2024-01-19 16:31:57', utc=True)
window_end   = pd.to_datetime('2024-01-19 17:01:57', utc=True)
mask_window = (trace_filtered['_ts_dt'] >= window_start) & (trace_filtered['_ts_dt'] <= window_end)
trace_window = trace_filtered[mask_window].copy()

# Merge global p95 into window rows so breaches are evaluated against global thresholds
trace_window = trace_window.merge(p95_df, on=['cmdb_id','trace_name'], how='left')

# 3) Find breaches in-window (value >= P95)
trace_window['is_breach'] = trace_window['value'] >= trace_window['p95']

# Per-group earliest breach timestamp and group window_max
# earliest breach timestamp (raw unix seconds)
breaches = trace_window[trace_window['is_breach']].copy()
earliest_breach_per_group = breaches.groupby(['cmdb_id','trace_name'], dropna=False)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_breach_timestamp'})
window_max_per_group = trace_window.groupby(['cmdb_id','trace_name'], dropna=False)['value'].max().reset_index().rename(columns={'value':'group_window_max'})

# Combine per-group info
per_group = p95_df.merge(window_max_per_group, on=['cmdb_id','trace_name'], how='left').merge(earliest_breach_per_group, on=['cmdb_id','trace_name'], how='left')
per_group['group_window_max'] = per_group['group_window_max'].round(6)

# 4) For each cmdb_id, find the earliest breach overall across the selected trace_names
rows = []
for cm in components:
    sub = per_group[per_group['cmdb_id'] == cm].copy()
    # drop groups with no breach (NaN earliest)
    sub_breaches = sub.dropna(subset=['earliest_breach_timestamp'])
    if sub_breaches.shape[0] == 0:
        rows.append({
            'cmdb_id': cm,
            'earliest_breach_timestamp': np.nan,
            'trace_name_of_first_breach': np.nan,
            'breach_value_at_first_breach': np.nan,
            'group_window_max_for_that_trace': np.nan
        })
    else:
        # find row with minimal earliest_breach_timestamp
        idx = sub_breaches['earliest_breach_timestamp'].astype(int).idxmin()
        chosen = sub_breaches.loc[idx]
        trace_nm = chosen['trace_name']
        ts_first = int(chosen['earliest_breach_timestamp'])
        # find the breach row(s) in breaches to get the actual value at that timestamp
        val_row = breaches[
            (breaches['cmdb_id']==cm) &
            (breaches['trace_name']==trace_nm) &
            (breaches['timestamp']==ts_first)
        ]
        # if multiple, take the max value at that timestamp (should typically be one)
        if val_row.shape[0] > 0:
            breach_val = float(val_row['value'].max())
        else:
            breach_val = np.nan
        rows.append({
            'cmdb_id': cm,
            'earliest_breach_timestamp': ts_first,
            'trace_name_of_first_breach': trace_nm,
            'breach_value_at_first_breach': round(breach_val,6) if not pd.isna(breach_val) else np.nan,
            'group_window_max_for_that_trace': chosen['group_window_max']
        })

cmdb_earliest_df = pd.DataFrame(rows)

# 5) Prepare up to top 20 trace anomaly rows within the window (value >= P95) sorted by timestamp ascending
anomalies_time_sorted = breaches[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=True).head(20).reset_index(drop=True)
anomalies_time_sorted['value'] = anomalies_time_sorted['value'].round(6)

# Final compact outputs: one-row-per-cmdb_id table and anomalies list
cmdb_earliest_df, anomalies_time_sorted

```
Out[5]:
```
Summary of trace-name breaches (restricted to the five services; window 2024-01-19 16:31:57 → 2024-01-19 17:01:57 UTC):

Per-service earliest breach (one row per cmdb_id; timestamps are raw unix seconds UTC)
- checkoutservice — earliest_breach_timestamp = 1705682040; trace = trace.from_checkoutservice.error_rate; breach_value = 0.0; group_window_max = 0.0
- currencyservice — earliest_breach_timestamp = 1705682040; trace = trace.from_checkoutservice.error_rate; breach_value = 0.0; group_window_max = 0.0
- emailservice — earliest_breach_timestamp = 1705682040; trace = trace.from_checkoutservice.duration_mean; breach_value = 0.000477; group_window_max = 0.000477
- productcatalogservice — earliest_breach_timestamp = 1705682040; trace = trace.from_checkoutservice.duration_mean; breach_value = 0.000029; group_window_max = 0.000029
- recommendationservice — earliest_breach_timestamp = 1705682040; trace = trace.from_frontendservice.error_rate; breach_value = 0.0; group_window_max = 0.0

Top trace anomaly rows within the window (value >= global P95), sorted by timestamp ascending (sample of notable entries from the returned list):
- 1705682040 — checkoutservice — trace.from_checkoutservice.error_rate — 0.0
- 1705682040 — productcatalogservice — trace.from_recommendationservice.error_rate — 0.0
- 1705682040 — checkoutservice — trace.to_emailservice.duration_mean — 0.000477
- 1705682040 — productcatalogservice — trace.from_root.duration_mean — 0.005755
- (many other entries at 1705682040 across services; most error_rate entries are 0.0)

Key observations and interpretation
- All five services show their earliest breach at the same unix timestamp 1705682040 (within the incident window).
- Many breaches are for error_rate series with P95 = 0, so value == 0 meets the ">= P95" rule; these produce many trivial/zero-value breaches and inflate breach counts.
- The meaningful non-zero early breaches are small duration increases (e.g., checkoutservice/productcatalogservice duration_mean values ~4.77e-4 to ~5.75e-3). The largest non-zero anomaly in the returned list is productcatalogservice trace.from_root.duration_mean = 0.005755 at timestamp 1705682040.
- Overall: the earliest detected anomalies across these trace types all occur at 1705682040 UTC, but many are zero-valued error_rate breaches due to P95 = 0; the notable non-zero trace anomalies at that time are small duration increases rather than large latency spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id  earliest_breach_timestamp                trace_name_of_first_breach  breach_value_at_first_breach  group_window_max_for_that_trace
0        checkoutservice                 1705682040     trace.from_checkoutservice.error_rate                      0.000000                         0.000000
1        currencyservice                 1705682040     trace.from_checkoutservice.error_rate                      0.000000                         0.000000
2           emailservice                 1705682040  trace.from_checkoutservice.duration_mean                      0.000477                         0.000477
3  productcatalogservice                 1705682040  trace.from_checkoutservice.duration_mean                      0.000029                         0.000029
4  recommendationservice                 1705682040     trace.from_frontendservice.error_rate                      0.000000                         0.000000,      timestamp                cmdb_id                                     trace_name     value
0   1705682040        checkoutservice          trace.from_checkoutservice.error_rate  0.000000
1   1705682040  productcatalogservice    trace.from_recommendationservice.error_rate  0.000000
2   1705682040        checkoutservice            trace.to_emailservice.duration_mean  0.000477
3   1705682040        checkoutservice               trace.to_emailservice.error_rate  0.000000
4   1705682040        checkoutservice    trace.to_productcatalogservice.duration_p95  0.000042
5   1705682040        checkoutservice      trace.to_productcatalogservice.error_rate  0.000000
6   1705682040        currencyservice          trace.from_checkoutservice.error_rate  0.000000
7   1705682040        currencyservice          trace.from_frontendservice.error_rate  0.000000
8   1705682040           emailservice       trace.from_checkoutservice.duration_mean  0.000477
9   1705682040           emailservice          trace.from_checkoutservice.error_rate  0.000000
10  1705682040        checkoutservice             trace.to_paymentservice.error_rate  0.000000
11  1705682040        checkoutservice   trace.to_productcatalogservice.duration_mean  0.000029
12  1705682040  productcatalogservice        trace.from_checkoutservice.duration_p95  0.000042
13  1705682040  productcatalogservice          trace.from_checkoutservice.error_rate  0.000000
14  1705682040  productcatalogservice        trace.from_frontendservice.duration_p95  0.000024
15  1705682040  productcatalogservice          trace.from_frontendservice.error_rate  0.000000
16  1705682040  productcatalogservice  trace.from_recommendationservice.duration_p95  0.000035
17  1705682040  recommendationservice      trace.to_recommendationservice.error_rate  0.000000
18  1705682040  productcatalogservice                  trace.from_root.duration_mean  0.005755
19  1705682040  productcatalogservice                     trace.from_root.error_rate  0.000000)```
```